In [ ]:
from pathlib import Path
import os

DATA_ROOT = Path(r"C:\REPLACE_WITH_LOCAL_DATA\Desktop\CCN_Spectrograms")
PAL_DIR = DATA_ROOT / "pal"
PER_DIR = DATA_ROOT / "per"

print("DATA_ROOT exists:", DATA_ROOT.exists(), DATA_ROOT)
print("PAL_DIR exists:", PAL_DIR.exists(), PAL_DIR)
print("PER_DIR exists:", PER_DIR.exists(), PER_DIR)


In [ ]:
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

def list_subjects(modality_dir: Path):
    return sorted([p for p in modality_dir.iterdir() if p.is_dir()])

def count_images_in_subject(subj_dir: Path):
    return sum(1 for p in subj_dir.rglob("*") if p.suffix.lower() in IMG_EXTS)

pal_subjects = list_subjects(PAL_DIR)
per_subjects = list_subjects(PER_DIR)

print("PAL subjects:", len(pal_subjects))
print("PER subjects:", len(per_subjects))

# show first 5 subject folder names
print("\nFirst 5 PAL subjects:", [p.name for p in pal_subjects[:5]])
print("First 5 PER subjects:", [p.name for p in per_subjects[:5]])

# image counts summary
pal_counts = [count_images_in_subject(s) for s in pal_subjects]
per_counts = [count_images_in_subject(s) for s in per_subjects]

print("\nPAL images per subject: min/mean/max =", min(pal_counts), sum(pal_counts)/len(pal_counts), max(pal_counts))
print("PER images per subject: min/mean/max =", min(per_counts), sum(per_counts)/len(per_counts), max(per_counts))

print("\nTotal PAL images:", sum(pal_counts))
print("Total PER images:", sum(per_counts))


In [ ]:
import random

def sample_filenames(modality_dir: Path, n_subjects=3, n_files=10):
    subjects = list_subjects(modality_dir)
    pick = random.sample(subjects, min(n_subjects, len(subjects)))
    out = {}
    for s in pick:
        files = [p for p in s.rglob("*") if p.suffix.lower() in IMG_EXTS]
        files = sorted(files)
        if len(files) == 0:
            continue
        out[s.name] = [f.name for f in random.sample(files, min(n_files, len(files)))]
    return out

print("Sample PAL filenames:")
pal_samples = sample_filenames(PAL_DIR, n_subjects=3, n_files=12)
for subj, names in pal_samples.items():
    print("\n", subj)
    for nm in names:
        print("  ", nm)

print("\n\nSample PER filenames:")
per_samples = sample_filenames(PER_DIR, n_subjects=3, n_files=12)
for subj, names in per_samples.items():
    print("\n", subj)
    for nm in names:
        print("  ", nm)


In [ ]:
import csv

OUT_CSV = DATA_ROOT / "labels_to_fill.csv"

rows = []
for modality, mdir in [("pal", PAL_DIR), ("per", PER_DIR)]:
    for subj_dir in list_subjects(mdir):
        subject_id = subj_dir.name
        for img_path in subj_dir.rglob("*"):
            if img_path.suffix.lower() in IMG_EXTS:
                rows.append([subject_id, modality, str(img_path), ""])  # region blank

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["subject_id", "modality", "image_path", "region_label"])
    w.writerows(rows)

print("✅ CSV created:", OUT_CSV)
print("Rows:", len(rows))
print("Min: open it in Excel, fill region_label with 8 region names, save.")


In [ ]:
import pandas as pd
import re
from pathlib import PureWindowsPath

csv_path = (r"C:\REPLACE_WITH_LOCAL_DATA\Desktop\CCN_Spectrograms/labels_to_fill.csv")  # uploaded file in this chat
df = pd.read_csv(csv_path)

def parse_pair(path):
    name = PureWindowsPath(path).name
    m = re.search(r"_spectrogram_(\d+)_(\d+)\.png$", name)
    if not m:
        return None, None
    return int(m.group(1)), int(m.group(2))

df["a"], df["b"] = zip(*df["image_path"].map(parse_pair))
print("Unique a values:", sorted(df["a"].dropna().unique()))
print("Unique b range:", int(df["b"].min()), "to", int(df["b"].max()))
print("\nCounts per a:")
print(df["a"].value_counts().sort_index())


In [ ]:
!pip install -q torch torchvision timm scikit-learn pillow


In [ ]:
import os, random, shutil
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torchvision import transforms
import timm
from sklearn.cluster import KMeans

DATA_ROOT = Path(r"C:\REPLACE_WITH_LOCAL_DATA\Desktop\CCN_Spectrograms")
OUT_ROOT = DATA_ROOT / "cluster_preview"
OUT_ROOT.mkdir(exist_ok=True)

IMG_EXTS = {".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp"}

# Collect a subset first (to keep it fast). We'll start with 2000 images.
all_imgs = []
for modality in ["pal","per"]:
    for subj in (DATA_ROOT/modality).iterdir():
        if subj.is_dir():
            for p in subj.rglob("*"):
                if p.suffix.lower() in IMG_EXTS:
                    all_imgs.append((modality, subj.name, p))

print("Total images found:", len(all_imgs))

random.seed(42)
sample_n = 2000  # start small; later we can scale up
sample = random.sample(all_imgs, sample_n)
print("Using sample:", len(sample))

# Pretrained feature extractor (fast + strong)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = timm.create_model("resnet50", pretrained=True, num_classes=0, global_pool="avg").to(device)
model.eval()

tfm = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

@torch.no_grad()
def embed_image(path):
    img = Image.open(path).convert("RGB")
    x = tfm(img).unsqueeze(0).to(device)
    feat = model(x).squeeze(0).cpu().numpy()
    return feat

# Extract embeddings
X = []
meta = []
for modality, subj, path in sample:
    X.append(embed_image(path))
    meta.append((modality, subj, str(path)))

X = np.vstack(X)
print("Embeddings shape:", X.shape)

# KMeans clustering into 8 groups
k = 8
km = KMeans(n_clusters=k, random_state=42, n_init="auto")
clusters = km.fit_predict(X)
print("Cluster counts:", np.bincount(clusters))

# Save preview images per cluster
for c in range(k):
    (OUT_ROOT / f"cluster_{c}").mkdir(parents=True, exist_ok=True)

# Copy 25 examples per cluster for you to inspect
per_cluster = 25
picked = {c:0 for c in range(k)}
for (modality, subj, p), c in zip(sample, clusters):
    if picked[c] < per_cluster:
        dst = OUT_ROOT / f"cluster_{c}" / f"{modality}_{subj}_{Path(p).name}"
        shutil.copy(p, dst)
        picked[c] += 1

print("✅ Saved preview folders here:", OUT_ROOT)
print("Open them and tell me what anatomical region each cluster corresponds to.")


In [ ]:
# If you already have these, you can skip this cell.
!pip -q install tensorflow scikit-learn pandas matplotlib pillow


In [ ]:
# # Cell 1 — Paths + basic checks

In [ ]:
import os, re, random
import numpy as np
import pandas as pd

DATA_ROOT = r"C:\REPLACE_WITH_LOCAL_DATA\Desktop\CCN_Spectrograms"
PAL_DIR = os.path.join(DATA_ROOT, "pal")
PER_DIR = os.path.join(DATA_ROOT, "per")

print("DATA_ROOT exists:", os.path.exists(DATA_ROOT), DATA_ROOT)
print("PAL_DIR exists:", os.path.exists(PAL_DIR), PAL_DIR)
print("PER_DIR exists:", os.path.exists(PER_DIR), PER_DIR)

assert os.path.isdir(PAL_DIR), "pal folder not found"
assert os.path.isdir(PER_DIR), "per folder not found"


In [ ]:
# Cell 2 — Build an index (image path, subject, modality label)

In [ ]:
from pathlib import Path

def list_subjects(folder):
    return sorted([d.name for d in Path(folder).iterdir() if d.is_dir()])

pal_subjects = list_subjects(PAL_DIR)
per_subjects = list_subjects(PER_DIR)

print("PAL subjects:", len(pal_subjects))
print("PER subjects:", len(per_subjects))
print("First 5 PAL subjects:", pal_subjects[:5])
print("First 5 PER subjects:", per_subjects[:5])

# Intersection check (should match)
common = sorted(set(pal_subjects).intersection(set(per_subjects)))
print("Common subjects:", len(common))
assert len(common) > 0, "No common subjects between pal and per"

def collect_rows(root_dir, modality_name, label_value):
    rows = []
    root = Path(root_dir)
    for subj_dir in root.iterdir():
        if not subj_dir.is_dir():
            continue
        subject = subj_dir.name
        for img_path in subj_dir.glob("*.png"):
            rows.append({
                "path": str(img_path),
                "subject": subject,
                "modality": modality_name,
                "label": int(label_value)
            })
    return rows

rows = []
rows += collect_rows(PAL_DIR, "pal", 0)
rows += collect_rows(PER_DIR, "per", 1)

df = pd.DataFrame(rows)
print("Total images:", len(df))
print(df["modality"].value_counts())
print("Unique subjects:", df["subject"].nunique())

# quick sanity check
print("Example rows:")
display(df.sample(5, random_state=42))


In [ ]:
# Cell 3 — Subject-wise split (train/val/test)

In [ ]:
# ============================================================
# FIXED SUBJECT-WISE SPLIT
# SAME SPLIT AS CNN, TRANSFORMER-ONLY AND CAPSULE-ONLY
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

train_subj = [
    'subject_17', 'subject_13', 'subject_20', 'subject_10',
    'subject_33', 'subject_01', 'subject_26', 'subject_06',
    'subject_12', 'subject_02', 'subject_38', 'subject_22',
    'subject_03', 'subject_30', 'subject_36', 'subject_04',
    'subject_35', 'subject_24', 'subject_32', 'subject_11',
    'subject_23', 'subject_19', 'subject_21', 'subject_08',
    'subject_15', 'subject_29'
]

val_subj = [
    'subject_37', 'subject_09', 'subject_31',
    'subject_28', 'subject_14', 'subject_07'
]

test_subj = [
    'subject_18', 'subject_16', 'subject_34',
    'subject_25', 'subject_27', 'subject_05'
]

train_df = df[df["subject"].isin(train_subj)].reset_index(drop=True)
val_df   = df[df["subject"].isin(val_subj)].reset_index(drop=True)
test_df  = df[df["subject"].isin(test_subj)].reset_index(drop=True)

print("=" * 60)
print("FIXED SUBJECT-WISE SPLIT")
print("=" * 60)

print("Subjects -> train/val/test:",
      len(train_subj), len(val_subj), len(test_subj))

print("Images -> train/val/test:",
      len(train_df), len(val_df), len(test_df))

print("\nTrain subjects:")
print(train_subj)

print("\nValidation subjects:")
print(val_subj)

print("\nTest subjects:")
print(test_subj)

print("\nTrain modality counts:")
print(train_df["modality"].value_counts())

print("\nValidation modality counts:")
print(val_df["modality"].value_counts())

print("\nTest modality counts:")
print(test_df["modality"].value_counts())

assert set(train_subj).isdisjoint(val_subj)
assert set(train_subj).isdisjoint(test_subj)
assert set(val_subj).isdisjoint(test_subj)

assert len(train_subj) == 26
assert len(val_subj) == 6
assert len(test_subj) == 6

print("\n✅ SAME CONTROLLED 26 / 6 / 6 SUBJECT SPLIT CONFIRMED")
print("✅ NO SUBJECT OVERLAP")

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

subjects = sorted(df["subject"].unique())

# 70% train, 15% val, 15% test by subject
train_subj, temp_subj = train_test_split(subjects, test_size=0.30, random_state=SEED)
val_subj, test_subj   = train_test_split(temp_subj, test_size=0.50, random_state=SEED)

train_df = df[df["subject"].isin(train_subj)].reset_index(drop=True)
val_df   = df[df["subject"].isin(val_subj)].reset_index(drop=True)
test_df  = df[df["subject"].isin(test_subj)].reset_index(drop=True)

print("Subjects -> train/val/test:", len(train_subj), len(val_subj), len(test_subj))
print("Images   -> train/val/test:", len(train_df), len(val_df), len(test_df))
print("Train modality counts:\n", train_df["modality"].value_counts())
print("Val modality counts:\n", val_df["modality"].value_counts())
print("Test modality counts:\n", test_df["modality"].value_counts())

# Safety: no subject overlap
assert set(train_subj).isdisjoint(val_subj)
assert set(train_subj).isdisjoint(test_subj)
assert set(val_subj).isdisjoint(test_subj)


In [ ]:
# Cell 4 — TensorFlow dataset loader (fast, no RAM blow-up)

In [ ]:
#import tensorflow as tf

#IMG_SIZE = (224, 224)   # strong default; change to (128,128) if you want faster
#BATCH_SIZE = 32
#AUTOTUNE = tf.data.AUTOTUNE

#def decode_img(path):
#    img = tf.io.read_file(path)
#    img = tf.image.decode_png(img, channels=1)              # grayscale
#    img = tf.image.resize(img, IMG_SIZE)
#    img = tf.cast(img, tf.float32) / 255.0
#    return img

#def augment(img):
#    # light augmentations for robustness
#    img = tf.image.random_brightness(img, max_delta=0.08)
#    img = tf.image.random_contrast(img, lower=0.90, upper=1.10)
#    return tf.clip_by_value(img, 0.0, 1.0)

#def make_ds(frame, training=False):
#    paths = frame["path"].values
#    labels = frame["label"].values.astype(np.int32)

#    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
#    if training:
#        ds = ds.shuffle(2000, seed=SEED, reshuffle_each_iteration=True)

#    ds = ds.map(lambda p, y: (decode_img(p), y), num_parallel_calls=AUTOTUNE)

#    if training:
#        ds = ds.map(lambda x, y: (augment(x), y), num_parallel_calls=AUTOTUNE)

#    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
#    return ds

#train_ds = make_ds(train_df, training=True)
#val_ds   = make_ds(val_df, training=False)
#test_ds  = make_ds(test_df, training=False)

#print("Train batches:", tf.data.experimental.cardinality(train_ds).numpy())
#print("Val batches:", tf.data.experimental.cardinality(val_ds).numpy())
#print("Test batches:", tf.data.experimental.cardinality(test_ds).numpy())


In [ ]:
import tensorflow as tf
import numpy as np

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def decode_img(path):
    img = tf.io.read_file(path)

    # FORCE grayscale = exactly 1 channel
    img = tf.image.decode_png(img, channels=1)

    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0

    # FORCE final shape
    img = tf.ensure_shape(img, [224, 224, 1])

    return img

def augment(img):
    img = tf.image.random_brightness(img, max_delta=0.08)
    img = tf.image.random_contrast(
        img,
        lower=0.90,
        upper=1.10
    )
    return tf.clip_by_value(img, 0.0, 1.0)

def make_ds(frame, training=False):
    paths = frame["path"].values
    labels = frame["label"].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(
            2000,
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        lambda p, y: (decode_img(p), y),
        num_parallel_calls=AUTOTUNE
    )

    if training:
        ds = ds.map(
            lambda x, y: (augment(x), y),
            num_parallel_calls=AUTOTUNE
        )

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

    return ds


train_ds = make_ds(train_df, training=True)
val_ds   = make_ds(val_df, training=False)
test_ds  = make_ds(test_df, training=False)


print("Train batches:",
      tf.data.experimental.cardinality(train_ds).numpy())

print("Val batches:",
      tf.data.experimental.cardinality(val_ds).numpy())

print("Test batches:",
      tf.data.experimental.cardinality(test_ds).numpy())


# IMPORTANT CHANNEL CHECK
for x_batch, y_batch in train_ds.take(1):
    print("Image batch shape:", x_batch.shape)
    print("Label batch shape:", y_batch.shape)

In [ ]:
#What changes in Phase 2 (new code)
# Do NOT run your old Capsule Cell 5



In [ ]:
# New Cell 5 (replace model): Capsule tokens + Transformer + classifier

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# -------------------------
# Transformer block
# -------------------------
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs, training=training)
        attn_output = self.drop1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1, training=training)
        ffn_output = self.drop2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# -------------------------
# Build Capsule + Transformer model
# (capsules are created as tokens, then transformer learns relations)
# -------------------------
def build_capsule_transformer_model(
    input_shape=(224, 224, 1),
    num_tokens=72,        # keep stable fixed token count
    token_dim=64,
    num_heads=4,
    ff_dim=128,
    transformer_layers=2,
    dropout=0.2
):
    inp = layers.Input(shape=input_shape)

    # grayscale -> 3ch
    x = layers.Lambda(lambda z: tf.image.grayscale_to_rgb(z))(inp)

    # backbone
    base = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(224,224,3)
    )
    base.trainable = False
    x = base(x)

    # Convert feature map into "capsule tokens"
    # (3x3x128 -> 72 tokens of dim 16 then project to token_dim)
    x = layers.Conv2D(128, kernel_size=3, strides=2, padding="valid")(x)   # (7,7,1280) -> (3,3,128)
    x = layers.Reshape((num_tokens, 16))(x)                                 # (3*3*8=72,16) stable

    # Project tokens to higher dimension for transformer
    x = layers.Dense(token_dim)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    # Transformer encoder stack
    for _ in range(transformer_layers):
        x = TransformerBlock(token_dim, num_heads, ff_dim, dropout)(x)

    # Pool tokens
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout)(x)

    # Binary classifier
    out = layers.Dense(1, activation="sigmoid")(x)

    model = Model(inp, out, name="CapsuleTransformer")
    return model

caps_trans_model = build_capsule_transformer_model()

caps_trans_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="acc"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

caps_trans_model.summary()


In [ ]:
for x_batch, y_batch in train_ds.take(1):
    print("Image batch shape:", x_batch.shape)
    print("Label batch shape:", y_batch.shape)

In [ ]:
# New Cell 6 (train)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=2, min_lr=1e-6),
]

history = caps_trans_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Get true test labels
y_true = np.concatenate([
    y.numpy() for _, y in test_ds
])

# Predict probabilities
y_probability = caps_trans_model.predict(
    test_ds,
    verbose=1
).ravel()

# Convert probabilities to class predictions
y_prediction = (y_probability >= 0.5).astype(int)

# Metrics
accuracy  = accuracy_score(y_true, y_prediction)
precision = precision_score(y_true, y_prediction)
recall    = recall_score(y_true, y_prediction)
f1        = f1_score(y_true, y_prediction)
auc       = roc_auc_score(y_true, y_probability)

tn, fp, fn, tp = confusion_matrix(
    y_true,
    y_prediction
).ravel()

print("=" * 55)
print("CAPSULE + TRANSFORMER — CONTROLLED TEST RESULTS")
print("=" * 55)

print(f"Accuracy : {accuracy:.6f}")
print(f"Precision: {precision:.6f}")
print(f"Recall   : {recall:.6f}")
print(f"F1-score : {f1:.6f}")
print(f"ROC-AUC  : {auc:.6f}")

print("\nConfusion Matrix")
print(f"TN = {tn}")
print(f"FP = {fp}")
print(f"FN = {fn}")
print(f"TP = {tp}")

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve

SAVE_DIR = r"C:\REPLACE_WITH_LOCAL_DATA\reliability_experiments\capsule_transformer"
os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
# 1. SAVE MODEL
# ============================================================
caps_trans_model.save(
    os.path.join(SAVE_DIR, "capsule_transformer_model.keras")
)

# ============================================================
# 2. SAVE MODEL SUMMARY
# ============================================================
with open(
    os.path.join(SAVE_DIR, "model_summary.txt"),
    "w",
    encoding="utf-8"
) as f:
    caps_trans_model.summary(
        print_fn=lambda x: f.write(x + "\n")
    )

# ============================================================
# 3. SAVE SUBJECT SPLIT
# ============================================================
subject_split = {
    "train_subjects": train_subj,
    "validation_subjects": val_subj,
    "test_subjects": test_subj
}

with open(
    os.path.join(SAVE_DIR, "subject_split.json"),
    "w"
) as f:
    json.dump(subject_split, f, indent=4)

# ============================================================
# 4. SAVE DATA MANIFESTS
# ============================================================
train_df.to_csv(
    os.path.join(SAVE_DIR, "train_manifest.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(SAVE_DIR, "validation_manifest.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(SAVE_DIR, "test_manifest.csv"),
    index=False
)

# ============================================================
# 5. SAVE TEST METRICS
# ============================================================
metrics_df = pd.DataFrame([{
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "roc_auc": auc,
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp
}])

metrics_df.to_csv(
    os.path.join(SAVE_DIR, "test_metrics.csv"),
    index=False
)

# ============================================================
# 6. SAVE TEST PREDICTIONS
# ============================================================
prediction_df = test_df.copy()

prediction_df["y_true"] = y_true
prediction_df["y_probability"] = y_probability
prediction_df["y_prediction"] = y_prediction

prediction_df.to_csv(
    os.path.join(SAVE_DIR, "test_predictions.csv"),
    index=False
)

# ============================================================
# 7. SAVE NUMPY ARRAYS
# ============================================================
np.save(
    os.path.join(SAVE_DIR, "y_true.npy"),
    y_true
)

np.save(
    os.path.join(SAVE_DIR, "y_probability.npy"),
    y_probability
)

np.save(
    os.path.join(SAVE_DIR, "y_prediction.npy"),
    y_prediction
)

# ============================================================
# 8. SAVE TRAINING HISTORY
# ============================================================
history_df = pd.DataFrame(history.history)

history_df.to_csv(
    os.path.join(SAVE_DIR, "training_history.csv"),
    index=False
)

# ============================================================
# 9. TRAINING / VALIDATION LOSS
# ============================================================
plt.figure(figsize=(7,5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(SAVE_DIR, "training_validation_loss.png"),
    dpi=300
)

plt.close()

# ============================================================
# 10. TRAINING / VALIDATION AUC
# ============================================================
plt.figure(figsize=(7,5))

plt.plot(history.history["auc"], label="Training AUC")
plt.plot(history.history["val_auc"], label="Validation AUC")

plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(SAVE_DIR, "training_validation_auc.png"),
    dpi=300
)

plt.close()

# ============================================================
# 11. CONFUSION MATRIX
# ============================================================
cm = confusion_matrix(y_true, y_prediction)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["PAL", "PER"]
)

disp.plot(values_format="d")

plt.tight_layout()

plt.savefig(
    os.path.join(SAVE_DIR, "confusion_matrix.png"),
    dpi=300
)

plt.close()

# ============================================================
# 12. ROC CURVE
# ============================================================
fpr, tpr, thresholds = roc_curve(
    y_true,
    y_probability
)

plt.figure(figsize=(7,5))

plt.plot(
    fpr,
    tpr,
    label=f"AUC = {auc:.6f}"
)

plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(SAVE_DIR, "roc_curve.png"),
    dpi=300
)

plt.close()

# ============================================================
# 13. SAVE EXPERIMENT CONFIG
# ============================================================
config = {
    "experiment": "Capsule + Transformer",
    "seed": 42,
    "image_size": [224, 224],
    "batch_size": 32,
    "train_subjects": 26,
    "validation_subjects": 6,
    "test_subjects": 6,
    "train_images": len(train_df),
    "validation_images": len(val_df),
    "test_images": len(test_df),
    "classification_threshold": 0.5,
    "backbone": "MobileNetV2",
    "backbone_pretrained": "ImageNet",
    "backbone_trainable": False,
    "token_dim": 64,
    "transformer_heads": 4,
    "transformer_ff_dim": 128,
    "transformer_layers": 2,
    "dropout": 0.2
}

with open(
    os.path.join(SAVE_DIR, "experiment_config.json"),
    "w"
) as f:
    json.dump(config, f, indent=4)

print("=" * 60)
print("CAPSULE + TRANSFORMER EXPERIMENT SAVED")
print("=" * 60)
print("Folder:")
print(SAVE_DIR)

print("\nSaved files:")
for file in sorted(os.listdir(SAVE_DIR)):
    print(file)

In [ ]:
# # Correct Cell 7 (Capsule Trans Model)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import numpy as np

# Predict probabilities
y_true = test_df["label"].values
y_prob = caps_trans_model.predict(test_ds, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("Test ROC-AUC:", roc_auc_score(y_true, y_prob))
print("\nClassification report:\n",
      classification_report(y_true, y_pred, target_names=["pal","per"]))

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix:\n", cm)

# Plot confusion matrix
plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix (pal vs per)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks([0,1], ["pal","per"])
plt.yticks([0,1], ["pal","per"])
for (i, j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha="center", va="center")
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(y_true, y_prob)
plt.figure()
plt.plot(fpr, tpr)
plt.plot([0,1], [0,1], linestyle="--")
plt.title("ROC Curve (Capsule – pal vs per)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()


In [ ]:
# Cell X — Save Confusion Matrix + ROC Curve (PNG) + metrics (TXT) into your results folder

import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, classification_report

# -------------------------
# 1) Choose output folder
# -------------------------
OUT_DIR = r"C:\REPLACE_WITH_LOCAL_DATA\Desktop\CCN_Spectrograms\optionB_results\phase2_capsule_transformer"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# 2) Predict on test
# -------------------------
y_true = test_df["label"].values
y_prob = caps_trans_model.predict(test_ds, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

# Metrics
auc = roc_auc_score(y_true, y_prob)
cm = confusion_matrix(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["pal", "per"])

print("Saved to:", OUT_DIR)
print("Test ROC-AUC:", auc)
print("Confusion matrix:\n", cm)
print("\nClassification report:\n", report)

# -------------------------
# 3) Save Confusion Matrix figure
# -------------------------
plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Confusion Matrix (pal vs per)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks([0, 1], ["pal", "per"])
plt.yticks([0, 1], ["pal", "per"])

for (i, j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha="center", va="center")

plt.tight_layout()
cm_path = os.path.join(OUT_DIR, "confusion_matrix_phase2.png")
plt.savefig(cm_path, dpi=300)
plt.close()

# -------------------------
# 4) Save ROC Curve figure
# -------------------------
fpr, tpr, _ = roc_curve(y_true, y_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {auc:.6f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("ROC Curve (Capsule + Transformer — pal vs per)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.tight_layout()
roc_path = os.path.join(OUT_DIR, "roc_curve_phase2.png")
plt.savefig(roc_path, dpi=300)
plt.close()

# -------------------------
# 5) Save text metrics
# -------------------------
txt_path = os.path.join(OUT_DIR, "metrics_phase2.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(f"Test ROC-AUC: {auc}\n\n")
    f.write("Confusion matrix:\n")
    f.write(np.array2string(cm))
    f.write("\n\nClassification report:\n")
    f.write(report)

print("✅ Saved:")
print(" -", cm_path)
print(" -", roc_path)
print(" -", txt_path)


In [ ]:
import matplotlib.pyplot as plt

models = ["CNN", "Capsule", "Capsule + Transformer"]
accuracy = [0.97, 0.97, 0.97]
f1 = [0.97, 0.97, 0.97]
auc = [0.9991, 0.9989, 0.9993]

x = range(len(models))

plt.figure(figsize=(8,5))
plt.bar(x, accuracy, width=0.25, label="Accuracy")
plt.bar([i+0.25 for i in x], f1, width=0.25, label="F1-score")
plt.bar([i+0.5 for i in x], auc, width=0.25, label="ROC-AUC")

plt.xticks([i+0.25 for i in x], models)
plt.ylabel("Score")
plt.title("Performance Comparison Across Models")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Loss curve
plt.figure(figsize=(7,5))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss (Capsule + Transformer)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# AUC curve
plt.figure(figsize=(7,5))
plt.plot(history.history["auc"], label="Train AUC")
plt.plot(history.history["val_auc"], label="Validation AUC")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title("Training and Validation AUC (Capsule + Transformer)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import os
import matplotlib.pyplot as plt

# -----------------------------
# Create results folder
# -----------------------------
save_dir = "results_capsule_transformer"
os.makedirs(save_dir, exist_ok=True)

# -----------------------------
# FIGURE 7(a): Training & Validation LOSS
# -----------------------------
plt.figure(figsize=(7, 5))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss (Capsule + Transformer)")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(
    os.path.join(save_dir, "Figure_7a_Loss_Capsule_Transformer.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# -----------------------------
# FIGURE 7(b): Training & Validation AUC
# -----------------------------
plt.figure(figsize=(7, 5))
plt.plot(history.history["auc"], label="Train AUC")
plt.plot(history.history["val_auc"], label="Validation AUC")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title("Training and Validation AUC (Capsule + Transformer)")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(
    os.path.join(save_dir, "Figure_7b_AUC_Capsule_Transformer.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("✅ Figure 7 saved successfully in:", save_dir)


In [ ]:
import os
import matplotlib.pyplot as plt

# -----------------------------
# Desktop path (Windows)
# -----------------------------
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
save_dir = os.path.join(desktop_path, "Capsule_Transformer_Results")
os.makedirs(save_dir, exist_ok=True)

# -----------------------------
# FIGURE 7(a): Training & Validation LOSS
# -----------------------------
plt.figure(figsize=(7, 5))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss (Capsule + Transformer)")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(
    os.path.join(save_dir, "Figure_7a_Loss_Capsule_Transformer.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# -----------------------------
# FIGURE 7(b): Training & Validation AUC
# -----------------------------
plt.figure(figsize=(7, 5))
plt.plot(history.history["auc"], label="Train AUC")
plt.plot(history.history["val_auc"], label="Validation AUC")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title("Training and Validation AUC (Capsule + Transformer)")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(
    os.path.join(save_dir, "Figure_7b_AUC_Capsule_Transformer.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("✅ Figure 7 saved on Desktop at:", save_dir)


# Experiment 1 — Transformer-Only Baseline

This experiment evaluates whether Transformer-based global modeling alone can explain the performance of the existing Capsule–Transformer framework. The same subject-wise train, validation, and test partitions are retained for fair comparison.

In [ ]:
# ============================================================
# Experiment 1 — Transformer-Only Baseline
# Same train/validation/test sets as Capsule–Transformer
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers, Model

# Reproducibility
SEED = 42
tf.keras.utils.set_random_seed(SEED)

class TransformerOnlyBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.2):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dropout(dropout),
            layers.Dense(embed_dim)
        ])

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, x, training=False):

        attn = self.attention(
            query=x,
            value=x,
            key=x,
            training=training
        )

        attn = self.dropout1(attn, training=training)
        x = self.norm1(x + attn)

        ffn = self.ffn(x, training=training)
        ffn = self.dropout2(ffn, training=training)

        return self.norm2(x + ffn)


def build_transformer_only_model(
    input_shape=(224, 224, 1),
    token_dim=64,
    num_heads=4,
    ff_dim=128,
    transformer_layers=2,
    dropout=0.2
):

    inp = layers.Input(shape=input_shape)

    # Convert grayscale spectrogram to RGB for MobileNetV2
    x = layers.Lambda(
        lambda z: tf.image.grayscale_to_rgb(z),
        name="gray_to_rgb"
    )(inp)

    # Same frozen backbone used in Capsule–Transformer
    base = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3)
    )

    base.trainable = False
    x = base(x, training=False)

    # --------------------------------------------------------
    # Transformer tokens directly from CNN feature map
    # NO capsule representation is used here
    # --------------------------------------------------------

    # MobileNetV2 output ≈ 7 × 7 × 1280
    x = layers.Conv2D(
        token_dim,
        kernel_size=1,
        padding="same",
        name="token_projection"
    )(x)

    # 7 × 7 spatial locations -> 49 Transformer tokens
    x = layers.Reshape(
        (49, token_dim),
        name="transformer_tokens"
    )(x)

    x = layers.LayerNormalization(
        epsilon=1e-6,
        name="token_norm"
    )(x)

    # Transformer encoder
    for i in range(transformer_layers):
        x = TransformerOnlyBlock(
            embed_dim=token_dim,
            num_heads=num_heads,
            ff_dim=ff_dim,
            dropout=dropout
        )(x)

    # Global token aggregation
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout)(x)

    output = layers.Dense(
        1,
        activation="sigmoid"
    )(x)

    return Model(
        inp,
        output,
        name="TransformerOnly"
    )


transformer_model = build_transformer_only_model()

transformer_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="acc"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

transformer_model.summary()

In [ ]:
# ============================================================
# Experiment 1 — Train Transformer-Only Baseline
# ============================================================

callbacks_transformer = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

history_transformer = transformer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=callbacks_transformer,
    verbose=1
)

In [ ]:
# ============================================================
# Experiment 1 — Evaluate Transformer-Only Baseline
# ============================================================

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# 1. Get true labels from the test dataset
# ------------------------------------------------------------
y_true = np.concatenate(
    [y.numpy().ravel() for _, y in test_ds],
    axis=0
).astype(int)

# ------------------------------------------------------------
# 2. Predict probabilities
# ------------------------------------------------------------
y_prob_transformer = transformer_model.predict(
    test_ds,
    verbose=1
).ravel()

# Binary prediction using threshold = 0.5
y_pred_transformer = (
    y_prob_transformer >= 0.5
).astype(int)

# ------------------------------------------------------------
# 3. Compute metrics
# ------------------------------------------------------------
transformer_accuracy = accuracy_score(
    y_true,
    y_pred_transformer
)

transformer_precision = precision_score(
    y_true,
    y_pred_transformer,
    zero_division=0
)

transformer_recall = recall_score(
    y_true,
    y_pred_transformer,
    zero_division=0
)

transformer_f1 = f1_score(
    y_true,
    y_pred_transformer,
    zero_division=0
)

transformer_auc = roc_auc_score(
    y_true,
    y_prob_transformer
)

transformer_cm = confusion_matrix(
    y_true,
    y_pred_transformer
)

# ------------------------------------------------------------
# 4. Print results
# ------------------------------------------------------------
print("\n========== Transformer-Only Test Results ==========")

print(f"Accuracy : {transformer_accuracy:.6f}")
print(f"Precision: {transformer_precision:.6f}")
print(f"Recall   : {transformer_recall:.6f}")
print(f"F1-score : {transformer_f1:.6f}")
print(f"ROC-AUC  : {transformer_auc:.6f}")

print("\nConfusion Matrix:")
print(transformer_cm)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred_transformer,
        target_names=["PAL", "PER"],
        digits=4
    )
)

In [ ]:
# ============================================================
# Save Transformer-Only Experiment Results
# ============================================================

import os
import numpy as np
import pandas as pd

RESULT_DIR = "reliability_experiments"
os.makedirs(RESULT_DIR, exist_ok=True)

# Save probabilities and labels
np.save(
    os.path.join(RESULT_DIR, "transformer_only_y_true.npy"),
    y_true
)

np.save(
    os.path.join(RESULT_DIR, "transformer_only_y_prob.npy"),
    y_prob_transformer
)

np.save(
    os.path.join(RESULT_DIR, "transformer_only_y_pred.npy"),
    y_pred_transformer
)

# Save summary metrics
transformer_results = pd.DataFrame({
    "Model": ["Transformer-only"],
    "Accuracy": [transformer_accuracy],
    "Precision": [transformer_precision],
    "Recall": [transformer_recall],
    "F1": [transformer_f1],
    "ROC_AUC": [transformer_auc],
    "TN": [transformer_cm[0, 0]],
    "FP": [transformer_cm[0, 1]],
    "FN": [transformer_cm[1, 0]],
    "TP": [transformer_cm[1, 1]]
})

transformer_results.to_csv(
    os.path.join(
        RESULT_DIR,
        "transformer_only_metrics.csv"
    ),
    index=False
)

# Save trained model
transformer_model.save(
    os.path.join(
        RESULT_DIR,
        "transformer_only_model.keras"
    )
)

print(transformer_results)
print("\nSaved successfully to:", RESULT_DIR)

In [ ]:
# ============================================================
# SAVE COMPLETE TRANSFORMER-ONLY EXPERIMENT
# Reproducibility + metrics + predictions + figures + metadata
# ============================================================

import os
import sys
import json
import random
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc
)

# ------------------------------------------------------------
# 1. Create experiment folder
# ------------------------------------------------------------
RESULT_DIR = "reliability_experiments/transformer_only"
os.makedirs(RESULT_DIR, exist_ok=True)

print("Saving experiment to:")
print(os.path.abspath(RESULT_DIR))

# ------------------------------------------------------------
# 2. Save reproducibility configuration
# ------------------------------------------------------------
experiment_config = {
    "experiment": "Transformer-only baseline",
    "seed": int(SEED),
    "image_size": list(IMG_SIZE),
    "batch_size": int(BATCH_SIZE),

    "total_subjects": int(df["subject"].nunique()),
    "total_images": int(len(df)),

    "train_subjects": int(len(train_subj)),
    "val_subjects": int(len(val_subj)),
    "test_subjects": int(len(test_subj)),

    "train_images": int(len(train_df)),
    "val_images": int(len(val_df)),
    "test_images": int(len(test_df)),

    "optimizer": "Adam",
    "initial_learning_rate": 1e-3,
    "loss": "binary_crossentropy",

    "max_epochs": 25,
    "early_stopping_monitor": "val_auc",
    "early_stopping_mode": "max",
    "early_stopping_patience": 5,

    "reduce_lr_monitor": "val_loss",
    "reduce_lr_factor": 0.5,
    "reduce_lr_patience": 2,
    "minimum_learning_rate": 1e-6,

    "decision_threshold": 0.5,

    "backbone": "MobileNetV2",
    "backbone_weights": "ImageNet",
    "backbone_trainable": False,

    "transformer_token_dim": 64,
    "transformer_heads": 4,
    "transformer_ff_dim": 128,
    "transformer_layers": 2,
    "dropout": 0.2,

    "tensorflow_version": tf.__version__,
    "python_version": sys.version,
    "platform": platform.platform()
}

with open(
    os.path.join(RESULT_DIR, "experiment_config.json"),
    "w"
) as f:
    json.dump(experiment_config, f, indent=4)

# ------------------------------------------------------------
# 3. Save exact subject split
# ------------------------------------------------------------
subject_split = {
    "train_subjects": [str(x) for x in train_subj],
    "validation_subjects": [str(x) for x in val_subj],
    "test_subjects": [str(x) for x in test_subj]
}

with open(
    os.path.join(RESULT_DIR, "subject_split.json"),
    "w"
) as f:
    json.dump(subject_split, f, indent=4)

# ------------------------------------------------------------
# 4. Save complete dataset manifests
# ------------------------------------------------------------
train_df.to_csv(
    os.path.join(RESULT_DIR, "train_manifest.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(RESULT_DIR, "validation_manifest.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(RESULT_DIR, "test_manifest.csv"),
    index=False
)

# ------------------------------------------------------------
# 5. Save model
# ------------------------------------------------------------
transformer_model.save(
    os.path.join(RESULT_DIR, "transformer_only_model.keras")
)

# ------------------------------------------------------------
# 6. Save model architecture summary
# ------------------------------------------------------------
summary_lines = []

transformer_model.summary(
    print_fn=lambda x: summary_lines.append(x)
)

with open(
    os.path.join(RESULT_DIR, "model_summary.txt"),
    "w",
    encoding="utf-8"
) as f:
    f.write("\n".join(summary_lines))

# ------------------------------------------------------------
# 7. Save training history
# ------------------------------------------------------------
history_df = pd.DataFrame(history_transformer.history)

history_df.to_csv(
    os.path.join(RESULT_DIR, "training_history.csv"),
    index=False
)

# ------------------------------------------------------------
# 8. Save predictions and probabilities
# ------------------------------------------------------------
np.save(
    os.path.join(RESULT_DIR, "y_true.npy"),
    y_true
)

np.save(
    os.path.join(RESULT_DIR, "y_probability.npy"),
    y_prob_transformer
)

np.save(
    os.path.join(RESULT_DIR, "y_prediction.npy"),
    y_pred_transformer
)

prediction_df = pd.DataFrame({
    "true_label": y_true,
    "predicted_probability": y_prob_transformer,
    "predicted_label": y_pred_transformer
})

prediction_df.to_csv(
    os.path.join(RESULT_DIR, "test_predictions.csv"),
    index=False
)

# ------------------------------------------------------------
# 9. Save test metrics
# ------------------------------------------------------------
transformer_results = pd.DataFrame({
    "Model": ["Transformer-only"],
    "Accuracy": [transformer_accuracy],
    "Precision": [transformer_precision],
    "Recall": [transformer_recall],
    "F1": [transformer_f1],
    "ROC_AUC": [transformer_auc],
    "TN": [transformer_cm[0, 0]],
    "FP": [transformer_cm[0, 1]],
    "FN": [transformer_cm[1, 0]],
    "TP": [transformer_cm[1, 1]]
})

transformer_results.to_csv(
    os.path.join(RESULT_DIR, "test_metrics.csv"),
    index=False
)

# ------------------------------------------------------------
# 10. Save confusion matrix figure
# ------------------------------------------------------------
plt.figure(figsize=(6, 5))

plt.imshow(transformer_cm)
plt.xticks([0, 1], ["PAL", "PER"])
plt.yticks([0, 1], ["PAL", "PER"])

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Transformer-Only Confusion Matrix")

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            transformer_cm[i, j],
            ha="center",
            va="center"
        )

plt.tight_layout()

plt.savefig(
    os.path.join(RESULT_DIR, "confusion_matrix.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 11. Save ROC curve
# ------------------------------------------------------------
fpr, tpr, thresholds = roc_curve(
    y_true,
    y_prob_transformer
)

roc_auc_value = auc(fpr, tpr)

plt.figure(figsize=(6, 5))

plt.plot(
    fpr,
    tpr,
    label=f"Transformer-only (AUC = {roc_auc_value:.6f})"
)

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Transformer-Only ROC Curve")
plt.legend(loc="lower right")

plt.tight_layout()

plt.savefig(
    os.path.join(RESULT_DIR, "roc_curve.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 12. Save training/validation loss curve
# ------------------------------------------------------------
plt.figure(figsize=(7, 5))

plt.plot(
    history_transformer.history["loss"],
    label="Training Loss"
)

plt.plot(
    history_transformer.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Transformer-Only Training and Validation Loss")
plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(RESULT_DIR, "training_validation_loss.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 13. Save training/validation AUC curve
# ------------------------------------------------------------
plt.figure(figsize=(7, 5))

plt.plot(
    history_transformer.history["auc"],
    label="Training AUC"
)

plt.plot(
    history_transformer.history["val_auc"],
    label="Validation AUC"
)

plt.xlabel("Epoch")
plt.ylabel("ROC-AUC")
plt.title("Transformer-Only Training and Validation AUC")
plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(RESULT_DIR, "training_validation_auc.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 14. Save exact random states
# ------------------------------------------------------------
np.save(
    os.path.join(RESULT_DIR, "numpy_random_state.npy"),
    np.random.get_state(),
    allow_pickle=True
)

# ------------------------------------------------------------
# 15. Final confirmation
# ------------------------------------------------------------
print("\n==============================================")
print("TRANSFORMER-ONLY EXPERIMENT SAVED SUCCESSFULLY")
print("==============================================")

print("\nResults:")
print(transformer_results.to_string(index=False))

print("\nSaved folder:")
print(os.path.abspath(RESULT_DIR))

print("\nSaved files:")
for file_name in sorted(os.listdir(RESULT_DIR)):
    print(" -", file_name)

In [ ]:
import json
from pathlib import Path

base = Path(r"C:\REPLACE_WITH_LOCAL_DATA\reliability_experiments")

folders = {
    "CNN baseline": base / "cnn_baseline",
    "Capsule-only": base / "capsule_only",
    "Capsule+Transformer": base / "capsule_transformer",
    "Transformer-only": base / "transformer_only",
}

splits = {}

for name, folder in folders.items():
    split_file = folder / "subject_split.json"

    if not split_file.exists():
        print(f"{name}: subject_split.json NOT FOUND")
        continue

    with open(split_file, "r") as f:
        splits[name] = json.load(f)

    print(f"\n{name}")
    print(splits[name])

print("\n" + "=" * 60)
print("COMPARISON")
print("=" * 60)

names = list(splits.keys())

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a = names[i]
        b = names[j]

        same = splits[a] == splits[b]

        print(f"{a} vs {b}: {'IDENTICAL ✅' if same else 'DIFFERENT ❌'}")

In [ ]:
# ============================================================
# CONTROLLED CNN BASELINE EXPERIMENT
# Uses EXACT same train/validation/test manifests as
# Transformer-only experiment.
#
# Output:
# C:\REPLACE_WITH_LOCAL_DATA\reliability_experiments\cnn_baseline
# ============================================================

import os
import json
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)

# ============================================================
# 1. PATHS
# ============================================================

ROOT = r"C:\REPLACE_WITH_LOCAL_DATA\reliability_experiments"

SOURCE_DIR = os.path.join(ROOT, "transformer_only")
SAVE_DIR = os.path.join(ROOT, "cnn_baseline")

os.makedirs(SAVE_DIR, exist_ok=True)

print("Source experiment:")
print(SOURCE_DIR)

print("\nSaving CNN baseline to:")
print(SAVE_DIR)


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


# ============================================================
# 3. LOAD TRANSFORMER CONFIG IF AVAILABLE
# ============================================================

config_path = os.path.join(SOURCE_DIR, "experiment_config.json")

source_config = {}

if os.path.exists(config_path):
    with open(config_path, "r") as f:
        source_config = json.load(f)

    print("\nTransformer experiment configuration loaded.")
    print(source_config)
else:
    print("\nexperiment_config.json not found.")
    print("Using default settings.")


# ------------------------------------------------------------
# Try to reuse important Transformer settings automatically
# ------------------------------------------------------------

def find_config_value(config, possible_keys, default):
    """
    Search recursively for a config value.
    """
    if not isinstance(config, dict):
        return default

    for key in possible_keys:
        if key in config:
            return config[key]

    for value in config.values():
        if isinstance(value, dict):
            found = find_config_value(value, possible_keys, None)
            if found is not None:
                return found

    return default


IMG_SIZE_RAW = find_config_value(
    source_config,
    [
        "image_size",
        "img_size",
        "input_size",
        "IMAGE_SIZE",
        "IMG_SIZE"
    ],
    224
)

if isinstance(IMG_SIZE_RAW, (list, tuple)):
    IMG_HEIGHT = int(IMG_SIZE_RAW[0])
    IMG_WIDTH = int(IMG_SIZE_RAW[1])
else:
    IMG_HEIGHT = int(IMG_SIZE_RAW)
    IMG_WIDTH = int(IMG_SIZE_RAW)

BATCH_SIZE = int(
    find_config_value(
        source_config,
        ["batch_size", "BATCH_SIZE"],
        32
    )
)

EPOCHS = int(
    find_config_value(
        source_config,
        ["epochs", "EPOCHS", "max_epochs"],
        30
    )
)

LEARNING_RATE = float(
    find_config_value(
        source_config,
        ["learning_rate", "lr", "LEARNING_RATE"],
        1e-4
    )
)

print("\nCNN experiment settings:")
print("Image size   :", (IMG_HEIGHT, IMG_WIDTH))
print("Batch size   :", BATCH_SIZE)
print("Epochs       :", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Seed         :", SEED)


# ============================================================
# 4. LOAD EXACT SAME MANIFESTS
# ============================================================

train_manifest_path = os.path.join(
    SOURCE_DIR,
    "train_manifest.csv"
)

val_manifest_path = os.path.join(
    SOURCE_DIR,
    "validation_manifest.csv"
)

test_manifest_path = os.path.join(
    SOURCE_DIR,
    "test_manifest.csv"
)

for p in [
    train_manifest_path,
    val_manifest_path,
    test_manifest_path
]:
    if not os.path.exists(p):
        raise FileNotFoundError(
            f"\nRequired manifest not found:\n{p}"
        )


train_df = pd.read_csv(train_manifest_path)
val_df = pd.read_csv(val_manifest_path)
test_df = pd.read_csv(test_manifest_path)

print("\nManifest columns:")
print(train_df.columns.tolist())

print("\nSamples:")
print("Train      :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))


# ============================================================
# 5. AUTOMATICALLY DETECT FILE-PATH AND LABEL COLUMNS
# ============================================================

def detect_path_column(df):

    candidates = [
        "filepath",
        "file_path",
        "path",
        "image_path",
        "filename",
        "file",
        "image"
    ]

    lower_map = {
        str(c).lower(): c
        for c in df.columns
    }

    for candidate in candidates:
        if candidate in lower_map:
            return lower_map[candidate]

    # Search values for common image extensions
    for col in df.columns:

        vals = df[col].astype(str).head(30)

        if vals.str.contains(
            r"\.(png|jpg|jpeg|bmp|tif|tiff)$",
            case=False,
            regex=True
        ).any():
            return col

    raise ValueError(
        "Could not automatically detect image path column."
    )


def detect_label_column(df):

    candidates = [
        "label",
        "class",
        "target",
        "y",
        "category"
    ]

    lower_map = {
        str(c).lower(): c
        for c in df.columns
    }

    for candidate in candidates:
        if candidate in lower_map:
            return lower_map[candidate]

    # Look for binary numeric column
    for col in df.columns:

        values = set(
            pd.to_numeric(
                df[col],
                errors="coerce"
            ).dropna().unique()
        )

        if values.issubset({0, 1}) and len(values) > 0:
            return col

    raise ValueError(
        "Could not automatically detect label column."
    )


PATH_COL = detect_path_column(train_df)
LABEL_COL = detect_label_column(train_df)

print("\nDetected columns:")
print("Image path column:", PATH_COL)
print("Label column     :", LABEL_COL)


# ============================================================
# 6. NORMALIZE LABELS
# ============================================================

def normalize_labels(df):

    out = df.copy()

    # Already numeric
    numeric = pd.to_numeric(
        out[LABEL_COL],
        errors="coerce"
    )

    if numeric.notna().all():

        out[LABEL_COL] = numeric.astype(int)

    else:

        # Expected classes PAL and PER
        label_map = {
            "pal": 0,
            "per": 1,
            "PAL": 0,
            "PER": 1
        }

        mapped = out[LABEL_COL].map(label_map)

        if mapped.isna().any():

            unique_values = sorted(
                out[LABEL_COL].astype(str).unique()
            )

            if len(unique_values) != 2:
                raise ValueError(
                    "Expected binary classification but found "
                    f"{unique_values}"
                )

            automatic_map = {
                unique_values[0]: 0,
                unique_values[1]: 1
            }

            print(
                "Automatic class mapping:",
                automatic_map
            )

            out[LABEL_COL] = (
                out[LABEL_COL]
                .astype(str)
                .map(automatic_map)
            )

        else:
            out[LABEL_COL] = mapped

    return out


train_df = normalize_labels(train_df)
val_df = normalize_labels(val_df)
test_df = normalize_labels(test_df)


# ============================================================
# 7. VERIFY FILES EXIST
# ============================================================

def check_files(df, name):

    paths = df[PATH_COL].astype(str).tolist()

    missing = [
        p for p in paths
        if not os.path.exists(p)
    ]

    print(
        f"{name}: {len(paths)} files, "
        f"{len(missing)} missing"
    )

    if missing:
        print("\nFirst missing files:")
        print(missing[:10])

        raise FileNotFoundError(
            f"{len(missing)} {name} image files were not found."
        )


check_files(train_df, "Train")
check_files(val_df, "Validation")
check_files(test_df, "Test")


# ============================================================
# 8. COPY EXACT SPLIT INFORMATION
# ============================================================

subject_split_source = os.path.join(
    SOURCE_DIR,
    "subject_split.json"
)

subject_split_dest = os.path.join(
    SAVE_DIR,
    "subject_split.json"
)

if os.path.exists(subject_split_source):
    shutil.copy2(
        subject_split_source,
        subject_split_dest
    )

    print(
        "\nExact subject_split.json copied "
        "to CNN experiment."
    )


# Save exact manifests used
train_df.to_csv(
    os.path.join(
        SAVE_DIR,
        "train_manifest.csv"
    ),
    index=False
)

val_df.to_csv(
    os.path.join(
        SAVE_DIR,
        "validation_manifest.csv"
    ),
    index=False
)

test_df.to_csv(
    os.path.join(
        SAVE_DIR,
        "test_manifest.csv"
    ),
    index=False
)


# ============================================================
# 9. BUILD TENSORFLOW DATASETS
# ============================================================

AUTOTUNE = tf.data.AUTOTUNE


def load_image(path, label):

    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_HEIGHT, IMG_WIDTH]
    )

    image = tf.cast(
        image,
        tf.float32
    )

    # MobileNetV2 preprocessing: scale to [-1, 1]
    image = tf.keras.applications.mobilenet_v2.preprocess_input(
        image
    )

    label = tf.cast(
        label,
        tf.float32
    )

    return image, label


def make_dataset(
    df,
    training=False
):

    paths = df[PATH_COL].astype(str).values
    labels = df[LABEL_COL].astype(int).values

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:

        ds = ds.shuffle(
            buffer_size=len(df),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(
        BATCH_SIZE
    )

    ds = ds.prefetch(
        AUTOTUNE
    )

    return ds


train_ds = make_dataset(
    train_df,
    training=True
)

val_ds = make_dataset(
    val_df,
    training=False
)

test_ds = make_dataset(
    test_df,
    training=False
)


# ============================================================
# 10. BUILD MOBILENETV2 CNN BASELINE
# ============================================================

print("\nBuilding MobileNetV2 CNN baseline...")


# Try ImageNet weights first.
# If weights already exist locally this needs no download.
try:

    backbone = MobileNetV2(
        input_shape=(
            IMG_HEIGHT,
            IMG_WIDTH,
            3
        ),
        include_top=False,
        weights="imagenet"
    )

    print(
        "MobileNetV2 ImageNet weights loaded."
    )

except Exception as e:

    print("\nWARNING:")
    print(
        "Could not load ImageNet weights."
    )
    print(e)

    print(
        "\nUsing MobileNetV2 with "
        "random initialization instead."
    )

    backbone = MobileNetV2(
        input_shape=(
            IMG_HEIGHT,
            IMG_WIDTH,
            3
        ),
        include_top=False,
        weights=None
    )


# Freeze pretrained feature extractor initially
backbone.trainable = False


inputs = layers.Input(
    shape=(
        IMG_HEIGHT,
        IMG_WIDTH,
        3
    ),
    name="input_image"
)

x = backbone(
    inputs,
    training=False
)

x = layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

x = layers.Dropout(
    0.30,
    name="dropout"
)(x)

outputs = layers.Dense(
    1,
    activation="sigmoid",
    name="binary_output"
)(x)


model = models.Model(
    inputs,
    outputs,
    name="CNN_MobileNetV2_Baseline"
)


model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.AUC(
            name="auc"
        )
    ]
)


model.summary()


# ============================================================
# 11. SAVE MODEL SUMMARY
# ============================================================

with open(
    os.path.join(
        SAVE_DIR,
        "model_summary.txt"
    ),
    "w"
) as f:

    model.summary(
        print_fn=lambda x: f.write(
            x + "\n"
        )
    )


# ============================================================
# 12. CALLBACKS
# ============================================================

model_path = os.path.join(
    SAVE_DIR,
    "cnn_baseline_model.keras"
)


callbacks = [

    ModelCheckpoint(
        model_path,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]


# ============================================================
# 13. TRAIN CNN
# ============================================================

print("\n==============================================")
print("TRAINING CNN BASELINE")
print("==============================================\n")


history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)


# ============================================================
# 14. LOAD BEST MODEL
# ============================================================

model = tf.keras.models.load_model(
    model_path
)


# ============================================================
# 15. TEST PREDICTIONS
# ============================================================

print("\nEvaluating CNN baseline on SAME test set...")


y_true = test_df[LABEL_COL].astype(int).values


y_probability = model.predict(
    test_ds,
    verbose=1
).ravel()


y_prediction = (
    y_probability >= 0.5
).astype(int)


# ============================================================
# 16. METRICS
# ============================================================

accuracy = accuracy_score(
    y_true,
    y_prediction
)

precision = precision_score(
    y_true,
    y_prediction,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_prediction,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_prediction,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_true,
    y_probability
)


tn, fp, fn, tp = confusion_matrix(
    y_true,
    y_prediction,
    labels=[0, 1]
).ravel()


metrics_df = pd.DataFrame(
    [{
        "Model": "CNN baseline (MobileNetV2)",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": roc_auc,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }]
)


print("\n==============================================")
print("CNN BASELINE RESULTS")
print("==============================================")
print(
    metrics_df.to_string(
        index=False
    )
)


# ============================================================
# 17. SAVE METRICS
# ============================================================

metrics_df.to_csv(
    os.path.join(
        SAVE_DIR,
        "test_metrics.csv"
    ),
    index=False
)


prediction_df = test_df.copy()

prediction_df["y_true"] = y_true
prediction_df["y_probability"] = y_probability
prediction_df["y_prediction"] = y_prediction


prediction_df.to_csv(
    os.path.join(
        SAVE_DIR,
        "test_predictions.csv"
    ),
    index=False
)


np.save(
    os.path.join(
        SAVE_DIR,
        "y_true.npy"
    ),
    y_true
)

np.save(
    os.path.join(
        SAVE_DIR,
        "y_probability.npy"
    ),
    y_probability
)

np.save(
    os.path.join(
        SAVE_DIR,
        "y_prediction.npy"
    ),
    y_prediction
)


# ============================================================
# 18. SAVE TRAINING HISTORY
# ============================================================

history_df = pd.DataFrame(
    history.history
)

history_df.to_csv(
    os.path.join(
        SAVE_DIR,
        "training_history.csv"
    ),
    index=False
)


# ============================================================
# 19. TRAINING / VALIDATION LOSS FIGURE
# ============================================================

plt.figure(
    figsize=(7, 5)
)

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(
    "CNN Baseline: Training and Validation Loss"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        SAVE_DIR,
        "training_validation_loss.png"
    ),
    dpi=300
)

plt.show()
plt.close()


# ============================================================
# 20. TRAINING / VALIDATION AUC FIGURE
# ============================================================

plt.figure(
    figsize=(7, 5)
)

plt.plot(
    history.history["auc"],
    label="Training AUC"
)

plt.plot(
    history.history["val_auc"],
    label="Validation AUC"
)

plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title(
    "CNN Baseline: Training and Validation AUC"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(
        SAVE_DIR,
        "training_validation_auc.png"
    ),
    dpi=300
)

plt.show()
plt.close()


# ============================================================
# 21. CONFUSION MATRIX FIGURE
# ============================================================

cm = confusion_matrix(
    y_true,
    y_prediction,
    labels=[0, 1]
)


plt.figure(
    figsize=(6, 5)
)

plt.imshow(
    cm,
    interpolation="nearest"
)

plt.title(
    "CNN Baseline Confusion Matrix"
)

plt.colorbar()

tick_marks = np.arange(2)

plt.xticks(
    tick_marks,
    ["PAL", "PER"]
)

plt.yticks(
    tick_marks,
    ["PAL", "PER"]
)

plt.xlabel("Predicted label")
plt.ylabel("True label")


threshold = cm.max() / 2.0

for i in range(2):
    for j in range(2):

        plt.text(
            j,
            i,
            str(cm[i, j]),
            horizontalalignment="center",
            verticalalignment="center"
        )


plt.tight_layout()

plt.savefig(
    os.path.join(
        SAVE_DIR,
        "confusion_matrix.png"
    ),
    dpi=300
)

plt.show()
plt.close()


# ============================================================
# 22. ROC CURVE
# ============================================================

fpr, tpr, thresholds = roc_curve(
    y_true,
    y_probability
)


plt.figure(
    figsize=(6, 5)
)

plt.plot(
    fpr,
    tpr,
    label=f"CNN baseline (AUC = {roc_auc:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "CNN Baseline ROC Curve"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        SAVE_DIR,
        "roc_curve.png"
    ),
    dpi=300
)

plt.show()
plt.close()


# ============================================================
# 23. SAVE EXPERIMENT CONFIG
# ============================================================

cnn_config = {

    "experiment": "cnn_baseline",

    "architecture":
        "MobileNetV2 CNN baseline",

    "controlled_split_source":
        SOURCE_DIR,

    "same_subject_split_as_transformer":
        True,

    "image_height":
        IMG_HEIGHT,

    "image_width":
        IMG_WIDTH,

    "batch_size":
        BATCH_SIZE,

    "epochs_requested":
        EPOCHS,

    "epochs_completed":
        len(history.history["loss"]),

    "learning_rate":
        LEARNING_RATE,

    "seed":
        SEED,

    "decision_threshold":
        0.5,

    "train_samples":
        len(train_df),

    "validation_samples":
        len(val_df),

    "test_samples":
        len(test_df),

    "path_column":
        PATH_COL,

    "label_column":
        LABEL_COL
}


with open(
    os.path.join(
        SAVE_DIR,
        "experiment_config.json"
    ),
    "w"
) as f:

    json.dump(
        cnn_config,
        f,
        indent=4
    )


# ============================================================
# 24. SAVE NUMPY RANDOM STATE
# ============================================================

# dtype=object avoids the ragged-array warning you saw earlier
np.save(
    os.path.join(
        SAVE_DIR,
        "numpy_random_state.npy"
    ),
    np.array(
        np.random.get_state(),
        dtype=object
    ),
    allow_pickle=True
)


# ============================================================
# 25. FINAL REPORT
# ============================================================

print("\n")
print("==============================================")
print("CNN BASELINE EXPERIMENT SAVED SUCCESSFULLY")
print("==============================================")

print("\nResults:")
print(
    metrics_df.to_string(
        index=False
    )
)

print("\nSaved folder:")
print(SAVE_DIR)

print("\nSaved files:")

for filename in sorted(
    os.listdir(SAVE_DIR)
):
    print(" -", filename)